# 🎙️ Meeting Transcript Generator — v10 local

Same outputs as `meeting_transcriber_v10.ipynb`, running on this laptop. No Colab, no Drive mount, no CUDA.

| v10 (Colab) | v10 local |
|---|---|
| `drive.mount()` | reads `G:\My Drive` directly |
| T4 GPU, hard-fails without one | CPU, 16 threads |
| `float16` | `int8` |
| whisperx + wav2vec2 alignment | faster-whisper native word timestamps |
| PyAV decoding | ffmpeg (PyAV is blocked by Smart App Control here) |
| HF token pasted in a cell | `.env` / environment |
| venv reinstalled every session | installed once |

**Kept from v10:** the 4-layer metadata extractor (GPS + original datetime), the A+B header, `.txt` + `.csv` beside each recording, skip-if-done.

**Run order:** Cell 1 → 2 → 3 → 4 → 5

## 📦 Cell 1 — One-time setup

Run once. Takes a few minutes; afterwards skip straight to Cell 2.

In [ ]:
# %pip install faster-whisper mutagen
#
# Speaker separation (optional, adds ~2.5 GB):
# %pip install torch --index-url https://download.pytorch.org/whl/cpu
# %pip install pyannote.audio
#
# ffmpeg is required and already installed here. If it ever goes missing:
#   winget install Gyan.FFmpeg

import shutil, sys
print('python  :', sys.version.split()[0])
print('ffmpeg  :', 'OK' if shutil.which('ffmpeg') else 'MISSING — winget install Gyan.FFmpeg')
print('exiftool:', 'OK' if shutil.which('exiftool') else 'not installed (optional — GPS layer 3)')
for mod in ('faster_whisper', 'mutagen', 'torch', 'pyannote.audio'):
    try:
        __import__(mod)
        print(f'{mod:15}: OK')
    except ImportError:
        print(f'{mod:15}: not installed')

## ⚙️ Cell 2 — Configuration *(EDIT HERE)*

The only cell you normally touch.

**Speed on this machine** — measured, 60s Vietnamese clip, int8/beam 5/VAD on:

| model | speed | 30-min recording |
|---|---|---|
| `tiny` | ~14× realtime | ~2 min |
| `small` | ~4.5× realtime | ~6 min |
| `large-v3-turbo` | ~2.7× realtime | ~11 min |

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, r"G:\My Drive\Second Brain Project\Transcriber")
from local_transcriber import Settings, Transcriber, scan, is_done
from local_transcriber.metadata import extract_audio_metadata

cfg = Settings(
    folder       = Path(r'G:\My Drive\Meet Recordings'),   # ← EDIT
    model        = 'large-v3-turbo',   # tiny | small | medium | large-v3-turbo | large-v3
    language     = 'vi',               # None = autodetect
    diarize      = True,               # needs HF_TOKEN + pyannote; degrades gracefully
    min_speakers = None,
    max_speakers = None,
    skip_done    = True,               # a file with .txt AND .csv is skipped
)

print(f'folder  : {cfg.folder}')
print(f'model   : {cfg.model}  (cpu / {cfg.compute_type} / {cfg.cpu_threads} threads)')
print(f'HF_TOKEN: {"set" if cfg.hf_token() else "not set — no speaker labels"}')

## 🔍 Cell 3 — Scan + metadata quick-check

Lists every recording with GPS and original datetime. Transcribes nothing — use it to see what you have before committing CPU time.

In [ ]:
files = scan(cfg.folder)
pending = [f for f in files if not is_done(f)]
print(f'📁 {len(files)} file(s) · ⏳ {len(pending)} pending · ✅ {len(files) - len(pending)} done\n')

print(f'{"File":<44}{"Dur":>9}  {"Original date":<22}{"GPS":<26}Done')
print('-' * 108)
for f in files:
    m = extract_audio_metadata(f)
    print(f'{f.name[:42]:<44}{m.get("duration_fmt", "N/A"):>9}  '
          f'{str(m.get("original_datetime", "N/A"))[:20]:<22}'
          f'{m.get("gps_display", "N/A")[:24]:<26}'
          f'{"✓" if is_done(f) else ""}')

## ▶️ Cell 4 — Transcribe

Models load once for the whole batch. Safe to interrupt — finished files are skipped on the next run, and `.txt`/`.csv` are written atomically so a half-written pair never counts as done.

In [ ]:
results = Transcriber(cfg).run()

## 👀 Cell 5 — Preview *(optional)*

In [ ]:
PREVIEW_INDEX = 0

done = [f for f in scan(cfg.folder) if is_done(f)]
if not done:
    print('Nothing transcribed yet — run Cell 4.')
else:
    p = done[PREVIEW_INDEX]
    print(p.with_suffix('.txt').read_text(encoding='utf-8')[:2500])

    try:
        import pandas as pd
        df = pd.read_csv(p.with_suffix('.csv'))
        print(f'\n=== CSV — {len(df)} rows ===')
        display(df[['Start_fmt', 'Speaker', 'Text']].head(15))
    except ImportError:
        print('(pip install pandas for the CSV preview)')

## 🖥️ Cell 6 — Terminal equivalent *(reference)*

The same pipeline without opening the notebook:

```powershell
cd "G:\My Drive\Second Brain Project\Transcriber"
# list recordings + metadata, transcribe nothing
python -m local_transcriber "G:\My Drive\Meet Recordings" --scan-only

# fast first pass, no speaker separation
python -m local_transcriber "G:\My Drive\Meet Recordings" --model small --no-diarize

# full quality, 2 known speakers
python -m local_transcriber "G:\My Drive\Meet Recordings" --min-speakers 2 --max-speakers 2
```